# Infer test filaments and write a submission CSV

This notebook loads the 12-epoch U-Net from `notebooks/train_unet.ipynb`, predicts each competition test image, and writes `/kaggle/working/submission.csv`.

It writes that file and stops. It does not submit it. There is no Kaggle submit call.

**Kaggle inputs to attach:** the competition data, the dataset `ignacioaguirrecandia/solar-code` (the repo zip, extracted as `Solar-Filament-Segmentation-main/`), and the output of the notebook `train-unet-r18-12ep-2026-10-31`. No cell needs editing.

Run the cells from the top. The seed is 0. A GPU is used when the notebook has one.


## 1. `src` and the checkpoint

`src` is the directory that contains `filament/train_data.py`. If that directory is already on `PYTHONPATH` (or already on `sys.path`), it is used. Otherwise the notebook walks `/kaggle/input` at any depth, skips every `competitions` directory, and stops at the first match.

The code dataset is nested. This path is one that the walk is meant to reach:

`/kaggle/input/datasets/ignacioaguirrecandia/solar-code/Solar-Filament-Segmentation-main/src`

`WEIGHTS_PATH` in the next cell defaults to the output of the 12-epoch training notebook. The path is written once, as `R18_12EP.weights_path` in `filament.checkpoint`, and the cell reads it from there:

`/kaggle/input/notebooks/ignacioaguirrecandia/train-unet-r18-12ep-2026-10-31/unet.pt`

Set it to another `.pt` file to load that one, or to `None` to search every `*.pt` under `/kaggle/working` and `/kaggle/input` (skipping the `competitions` tree). With `None`, one file is used; several files use `unet.pt` when exactly one candidate has that name; otherwise the notebook stops and lists every candidate.

**Checkpoint guard.** The first U-Net submission scored LB 0.04 because it loaded 3-epoch weights by mistake. Before predicting, this notebook compares the loaded checkpoint with the expected one (`filament.checkpoint`: `epochs` 12 and `tensor_sha256` `70c341c9…9a04`) and stops with an error on any difference. `ALLOW_UNVERIFIED_WEIGHTS = True` turns the error into a printed warning. Leave it `False` for a real submission.

Kaggle extracts an uploaded `.pt` because `torch.save` writes a zip.
`unet.pt` becomes a directory that contains `unet/data.pkl`. That
directory is not a checkpoint. The loader re-zips it to a real file
under `/kaggle/working` and prints that path. If the folder is not one
torch archive, the notebook stops and tells you to set `WEIGHTS_PATH`
to a real `.pt` file.

The notebook prints a fingerprint of the checkpoint it loads: the resolved path, the file size in bytes, the MD5 of those file bytes, epochs, encoder, and image size from the checkpoint, and a hash of the `state_dict` tensor bytes. A field that is absent is printed as `not stored`. The file MD5 of a re-zip is not comparable to the original file MD5. The tensor hash is the one to compare.

The file is the dict the training notebook saves (`state_dict`, `encoder_name`, `encoder_weights`, `smp_version`, `side`, `seed`, `epochs`, `batch_size`, `in_channels`). This notebook only reads it.


In [ ]:
import os
import sys
import zipfile
from pathlib import Path

# False: stop if the checkpoint is not the 12-epoch weights (epochs and
# tensor SHA-256 are checked). True only prints the mismatch. Keep False
# for a real submission.
ALLOW_UNVERIFIED_WEIGHTS = False

# Postprocess settings from the 2026-11-01 holdout sweep (docs/RESULTS.md).
# min_area is for a 2048x2048 frame. close_size 0 means no closing.
MIN_AREA = 400
CLOSE_SIZE = 0

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")


def _has_train_data(directory: Path) -> bool:
    return (directory / "filament" / "train_data.py").is_file()


def _src_from_entries(entries: list[str]) -> Path | None:
    for part in entries:
        if part == "":
            continue
        directory = Path(part)
        if _has_train_data(directory):
            return directory
    return None


def _walk_input(root: Path):
    """Yield `(directory, filenames)` under `root`, skipping `competitions`."""
    if not root.is_dir():
        return
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [name for name in sorted(dirnames) if name != "competitions"]
        yield Path(dirpath), filenames


def _find_src() -> Path:
    raw = os.environ.get("PYTHONPATH", "")
    found = _src_from_entries(raw.split(os.pathsep))
    if found is None:
        found = _src_from_entries(list(sys.path))
    if found is None:
        for directory, _filenames in _walk_input(KAGGLE_INPUT):
            if _has_train_data(directory):
                found = directory
                break
    if found is None:
        raise FileNotFoundError(
            "Could not find the directory that contains "
            "filament/train_data.py. Put that src directory on PYTHONPATH, "
            "or attach the solar-code dataset under /kaggle/input outside "
            "competitions. One nested path the walk covers is "
            "/kaggle/input/datasets/ignacioaguirrecandia/"
            "solar-code/Solar-Filament-Segmentation-main/src."
        )
    return found


def _pt_under(root: Path) -> list[Path]:
    """Every ``*.pt`` file or directory under ``root``.

    A directory counts because Kaggle extracts an uploaded ``.pt`` zip
    into a folder of that name. ``competitions`` is not walked.
    """
    found: list[Path] = []
    if not root.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            if name.endswith(".pt"):
                found.append(directory / name)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
    return found


def _weight_candidates() -> list[Path]:
    """``.pt`` files and extracted archives, working then input."""
    return _pt_under(KAGGLE_WORKING) + _pt_under(KAGGLE_INPUT)


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )


WEIGHTS_SOURCE: Path | None = None


def _resolve_weights(explicit: Path | str | None) -> Path | None:
    """Return the checkpoint ``explicit`` names, or one discovered ``*.pt``.

    One discovered file or extracted directory is used, including
    ``unet_12ep.pt`` when it is the only ``.pt``. Several matches use
    ``unet.pt`` when exactly one candidate has that name. Several matches
    and no single ``unet.pt`` raise and list every candidate. An extracted
    torch archive is re-zipped under ``/kaggle/working`` and that file is
    returned. No match returns ``None``.
    """
    global WEIGHTS_SOURCE
    WEIGHTS_SOURCE = None
    if explicit is not None:
        path = Path(explicit)
        if not path.exists():
            raise FileNotFoundError(
                f"WEIGHTS_PATH is set to {path}, but that file does not exist. "
                "Set WEIGHTS_PATH to an existing .pt file."
            )
        WEIGHTS_SOURCE = path
        path = _materialize_checkpoint(path)
        print(f"loaded weights={path}")
        return path
    candidates = _weight_candidates()
    if not candidates:
        return None
    if len(candidates) == 1:
        WEIGHTS_SOURCE = candidates[0]
        chosen = _materialize_checkpoint(candidates[0])
        print(f"loaded weights={chosen}")
        return chosen
    named = [path for path in candidates if path.name == "unet.pt"]
    if len(named) == 1:
        WEIGHTS_SOURCE = named[0]
        chosen = _materialize_checkpoint(named[0])
        print(f"Several .pt files were found. Using the one named unet.pt: {chosen}")
        print(f"loaded weights={chosen}")
        return chosen
    listed = "\n".join(f"  {path}" for path in candidates)
    if named:
        detail = "Several files named unet.pt were found."
    else:
        detail = "Several .pt files were found and none is named unet.pt."
    raise RuntimeError(f"{detail} Set WEIGHTS_PATH to the file to load.\n{listed}")


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


SRC = _find_src()
_put_on_path(SRC)
print(f"src={SRC}")

from filament.checkpoint import R18_12EP

# Default: the 12-epoch ResNet-18 512 px weights (Kaggle notebook output). The
# path is written once, in filament.checkpoint (R18_12EP.weights_path).
# Set another .pt path to load that checkpoint, or None to search
# /kaggle/working and /kaggle/input (skipping competitions).
# A directory from Kaggle extracting a .pt zip is re-zipped under /kaggle/working.
WEIGHTS_PATH = R18_12EP.weights_path

## 2. Install the segmentation library

`segmentation-models-pytorch` **0.6.0** (MIT). The encoder is `resnet18` with `encoder_weights="imagenet"`, the same constructor as training. Those weights are the torchvision checkpoint `resnet18-5c106cde.pth` (BSD-3).

Internet has to be on for the install and for that download. A Hugging Face Hub warning during the download is expected. No Hugging Face token is set. Kaggle already has PyTorch. This cell does not install PyTorch.


In [ ]:
import subprocess

SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Keep encoder_weights='imagenet'. Kaggle already has PyTorch."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 3. Load the training checkpoint

The checkpoint dict is the one `notebooks/train_unet.ipynb` writes. `state_dict` is loaded into a new `smp.Unet` (`resnet18`, `imagenet`, 3 input channels, 1 class). The side recorded in the file has to be 512.

The checkpoint fingerprint is printed, then checked with `filament.checkpoint.verify_checkpoint` before the network is built. A checkpoint whose `epochs` is not 12 or whose `tensor_sha256` is not `70c341c95596a62160983e335d176bad14e29e0755d9bda3e66787d7b9079a04` stops the notebook unless `ALLOW_UNVERIFIED_WEIGHTS` is `True`.

Seed 0 is set before the network is built. CUDA is used when `torch.cuda.is_available()` is true.


In [ ]:
import hashlib
import pickle
import random

import numpy as np
import torch

from filament.checkpoint import R18_12EP, verify_and_report
from filament.checkpoint import tensor_sha256 as _tensor_sha256


def _stored(checkpoint: dict, key: str) -> str:
    if key not in checkpoint or checkpoint[key] is None:
        return "not stored"
    return str(checkpoint[key])


def _checkpoint_fingerprint(checkpoint: dict) -> dict:
    """Fields `verify_checkpoint` compares: stored epochs and tensor hash."""
    state = checkpoint.get("state_dict")
    return {
        "epochs": checkpoint.get("epochs"),
        "tensor_sha256": _tensor_sha256(state) if isinstance(state, dict) else None,
    }


def _load_checkpoint_dict(path: Path) -> tuple[dict | None, str | None]:
    try:
        try:
            loaded = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            loaded = torch.load(path, map_location="cpu")
    except (OSError, RuntimeError, ValueError, EOFError, pickle.UnpicklingError) as exc:
        return None, str(exc)
    if isinstance(loaded, dict):
        return loaded, None
    return None, None


def _print_fingerprint(
    label: str,
    path: Path,
    checkpoint: dict | None,
    error: str | None = None,
) -> None:
    data = path.read_bytes()
    print(f"fingerprint {label}")
    print(f"  path={path}")
    print(f"  bytes={len(data)}")
    print(f"  file_md5={hashlib.md5(data).hexdigest()}")
    if error is not None:
        print(f"  error={error}")
    if not isinstance(checkpoint, dict):
        print("  epochs=not stored")
        print("  encoder=not stored")
        print("  image_size=not stored")
        print("  tensor_sha256=not stored")
        return
    print(f"  epochs={_stored(checkpoint, 'epochs')}")
    print(f"  encoder={_stored(checkpoint, 'encoder_name')}")
    print(f"  image_size={_stored(checkpoint, 'side')}")
    state = checkpoint.get("state_dict")
    if isinstance(state, dict):
        print(f"  tensor_sha256={_tensor_sha256(state)}")
    else:
        print("  tensor_sha256=not stored")


def _print_fingerprint_note() -> None:
    print(
        "File MD5 of a re-zip is not comparable to the original file MD5. "
        "Compare tensor_sha256."
    )


SEED = 0


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _load_training_checkpoint(path: Path) -> dict:
    """Read the dict written by `notebooks/train_unet.ipynb`."""
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location="cpu")
    if not isinstance(checkpoint, dict) or "state_dict" not in checkpoint:
        raise RuntimeError(
            f"{path} is not the checkpoint written by notebooks/train_unet.ipynb. "
            "Expected a dict with a state_dict key."
        )
    encoder_name = checkpoint.get("encoder_name", "resnet18")
    if encoder_name != "resnet18":
        raise RuntimeError(
            f"{path} records encoder_name={encoder_name!r}. This notebook loads resnet18."
        )
    encoder_weights = checkpoint.get("encoder_weights", "imagenet")
    if encoder_weights != "imagenet":
        raise RuntimeError(
            f"{path} records encoder_weights={encoder_weights!r}. "
            "This notebook loads imagenet."
        )
    saved_side = int(checkpoint.get("side", 512))
    if saved_side != 512:
        raise RuntimeError(
            f"{path} records side={saved_side}. This notebook predicts at 512."
        )
    saved_channels = int(checkpoint.get("in_channels", 3))
    if saved_channels != 3:
        raise RuntimeError(
            f"{path} records in_channels={saved_channels}. This notebook uses 3."
        )
    saved_smp = checkpoint.get("smp_version")
    if saved_smp is not None and saved_smp != smp.__version__:
        raise RuntimeError(
            f"{path} was saved with segmentation-models-pytorch {saved_smp}. "
            f"This notebook has {smp.__version__}."
        )
    return checkpoint


def _build_model() -> torch.nn.Module:
    try:
        return smp.Unet(
            encoder_name="resnet18",
            encoder_weights="imagenet",
            in_channels=3,
            classes=1,
        )
    except Exception as exc:
        raise RuntimeError(
            "resnet18 ImageNet weights did not load. Use encoder_weights='imagenet'. "
            "A Hugging Face Hub warning on that download is expected. "
            f"Original error: {exc}"
        ) from exc


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"device={torch.cuda.get_device_name(0)}")
else:
    print("device=cpu")

WEIGHTS_PATH = _resolve_weights(WEIGHTS_PATH)
if WEIGHTS_PATH is None:
    raise FileNotFoundError(
        "No U-Net weights were found. Set WEIGHTS_PATH to a .pt file, "
        "or attach one .pt under /kaggle/working or under /kaggle/input "
        "outside competitions."
    )
checkpoint = _load_training_checkpoint(WEIGHTS_PATH)
try:
    _print_fingerprint_note()
    _print_fingerprint("loaded", WEIGHTS_PATH, checkpoint)
except Exception as exc:  # noqa: BLE001
    print(f"fingerprint loaded error={exc}")
# Raises CheckpointMismatchError on the wrong weights (for example the
# 3-epoch ones behind LB 0.04) unless ALLOW_UNVERIFIED_WEIGHTS is True.
verify_and_report(
    _checkpoint_fingerprint(checkpoint),
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=R18_12EP,
)
model = _build_model()
model.load_state_dict(checkpoint["state_dict"])
model = model.to(device)
model.eval()
print(f"weights={WEIGHTS_PATH}")
print(f"seed={SEED}")

## 4. Predict the test images and write the CSV

Test images, 180 JPEG files, and not the train set:

`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images`

Each image is read at 2048. The network input is the same 512 area average used in training (512 divides 2048), with the gray band repeated to 3 channels and ImageNet normalization. The semantic mask is the probability at 0.5, nearest-neighbor upsampled to 2048. At this scale each predicted pixel becomes a 4×4 block.

Instances are `filament.instances.labels_from_semantic` on that 2048 mask and the 2048 grayscale image. The area cut is `MIN_AREA` (400 on a 2048 frame) and the close size is `CLOSE_SIZE` (0, no closing). Both come from the 2026-11-01 holdout sweep (`docs/RESULTS.md`) and match the library defaults.

`filament.submission.write_submission` writes `/kaggle/working/submission.csv` with columns `filament_id` and `segmentation_rle`. `validate_submission` (`filament.validate`) then checks that file. This cell does not submit the file.


In [ ]:
import cv2

from filament.baseline import iter_filament_counts
from filament.instances import labels_from_semantic, starting_min_area
from filament.submission import filament_id, write_submission
from filament.train_data import downscale_pair
from filament.validate import validate_submission

TEST_IMAGES = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026/test/test_images"
)
SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
FULL_SIDE = 2048

# torchvision ImageNet normalization. The resnet18 weights were trained with it.
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


def _test_paths() -> list[Path]:
    if not TEST_IMAGES.is_dir():
        raise FileNotFoundError(
            f"Test images were not found at {TEST_IMAGES}. Train images are not read."
        )
    paths = sorted(
        path
        for path in TEST_IMAGES.iterdir()
        if path.is_file() and path.suffix.lower() == ".jpeg"
    )
    if not paths:
        raise FileNotFoundError(f"No .jpeg files in {TEST_IMAGES}")
    return paths


def _read_gray(path: Path) -> np.ndarray:
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(path)
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise ValueError(f"{path.name} is not a uint8 grayscale image")
    if gray.shape != (FULL_SIDE, FULL_SIDE):
        raise ValueError(
            f"{path.name} is {gray.shape[0]}x{gray.shape[1]}, "
            f"expected {FULL_SIDE}x{FULL_SIDE}"
        )
    return np.ascontiguousarray(gray)


def _network_tensor(gray: np.ndarray) -> torch.Tensor:
    """Area-average to 512 and normalize the way training does."""
    blank = np.zeros(gray.shape, dtype=np.uint8)
    small, _mask = downscale_pair(gray, blank, side=512)
    scaled = small.astype(np.float32) / 255.0
    channels = np.stack((scaled, scaled, scaled), axis=0)
    normalized = (channels - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(normalized))


def _semantic_full(gray: np.ndarray) -> np.ndarray:
    batch = _network_tensor(gray).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(batch)
        small = (torch.sigmoid(logits)[0, 0] >= 0.5).to(dtype=torch.uint8)
    mask = small.detach().cpu().numpy()
    full = cv2.resize(
        mask,
        (FULL_SIDE, FULL_SIDE),
        interpolation=cv2.INTER_NEAREST,
    )
    return np.ascontiguousarray(full)


def _iter_rows(paths: list[Path]):
    total = len(paths)
    for index, path in enumerate(paths, start=1):
        gray = _read_gray(path)
        semantic = _semantic_full(gray)
        labels = labels_from_semantic(
            semantic, gray, min_area=MIN_AREA, close_size=CLOSE_SIZE
        )
        count = 0
        for number, counts in iter_filament_counts(labels):
            count += 1
            yield filament_id(path.stem, number), counts
        del gray, semantic, labels
        if index == 1 or index % 20 == 0 or index == total:
            print(f"{index}/{total} {path.name} filaments={count}")


def _write_predictions() -> None:
    paths = _test_paths()
    area_cut = MIN_AREA
    if MIN_AREA != starting_min_area(FULL_SIDE, FULL_SIDE):
        print("MIN_AREA differs from the library default")
    print(f"test images={len(paths)} directory={TEST_IMAGES}")
    print(
        f"predict side=512 upsample={FULL_SIDE} nearest area_cut={area_cut} close_size={CLOSE_SIZE}"
    )
    print("train images are not read")
    SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    written = 0

    def _counted():
        nonlocal written
        for row in _iter_rows(paths):
            written += 1
            yield row

    write_submission(SUBMISSION_PATH, _counted())
    validate_submission(SUBMISSION_PATH)
    print(f"wrote {written} rows to {SUBMISSION_PATH}")
    print("columns=filament_id,segmentation_rle")
    print("This notebook wrote the CSV. It did not submit it.")


_write_predictions()